<a href="https://colab.research.google.com/github/GMISSAGLIA/GM_PyLab/blob/Main/python_blp_api_multi_ptf_request.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **Bloomberg API: A Python Example for Multiple Portfolio Data Requests**


If you want to automate data extraction from Bloomberg, you have to use the Bloomberg API, which allows you to interface with Bloomberg programmatically. You can use multiple programming languages according to your preferences; naturally, as a Windows and Office user, my first choices have been VBA in Excel and MS Access, and .NET C# to implement .dll or .xll in different ways. Eventually, I discovered Python, and it surprised me. Pandas DataFrames are a natural fit for retrieving Bloomberg data, so you need less mapping code to get your results easily.


To executee the following code you need just an excel file as input, [here you can download the template input file](https://github.com/GMISSAGLIA/GM_PyLab/blob/Main/GITHUB_BLP_API_INPUT.xlsx?raw=true)
which you have to fill with your Bloomberg Portfolios list (PTF_CODES sheet) and the list of the fields (FIELDS sheet) you want to request from Bloomberg for each of the securities in each portfolio. Just download the file, input your data, and save it in your preferred location.

Before running the code, you need to install:

1.    [The Bloomberg Official Python API ](https://www.bloomberg.com/professional/support/api-library/)
2.  and then [the Bloomberg XBBG API ](https://xbbg.readthedocs.io/en/latest/) which provides easier access to the Official Bloomberg API (it acts as a second API layer that improves your programming experience)

If you are using the Anaconda distribution you can install the  Bloomberg Python API through ‘conda’ as reported here:

1.   [CondaBloomberg API] (https://anaconda.org/conda-forge/blpapi)
2.  then install the xbbg package as reported here :
[ - xbbg - Bloomberg API](https://anaconda.org/conda-forge/xbbg)


Note that local data usage must be compliant with Bloomberg Datafeed Addendum (full description in DAPI<GO>):
To access Bloomberg data via the API (and use that data in Microsoft Excel), your company must sign the 'Datafeed Addendum' to the Bloomberg Agreement. This legally binding contract describes the terms and conditions of your use of the data and information available via the API (the "Data"). The most fundamental requirement regarding your use of Data is that it cannot leave the local PC you use to access the BLOOMBERG PROFESSIONAL service.


In [ ]:
import numpy as np
import pandas as pd
import blpapi
from xbbg import blp

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [ ]:
class GM_BLP_PTF:

    def DataFrameInit(self):
        self.PTF_CODES = pd.DataFrame()
        self.BLP_FIELDS = pd.DataFrame()
        self.DF_PORTFOLIOS = pd.DataFrame()
        self.DF_SECURITIES = pd.DataFrame()
        self.DF_PTF_ALL = pd.DataFrame()
        self.Securities = list()
        self.Ptfcodes = list()
        self.Fields = list()
        self.Overrides = {}

    def __init__(self,WDIR=r'C:\Dati\BLP_API_DATA'):
        self.DataFrameInit()
        self._wdir = WDIR

    @property
    def wdir(self):
        return self._wdir

    @wdir.setter
    def wdir(self, value):
        self._wdir = value

    def remove(self, string):
	     return string.replace(" ", "")


    def Load_Input(self, str_file_name = r'\BLP_API_INPUT.xlsx'):
        self.InputFile = self.wdir + str_file_name

        self.PTF_CODES = pd.read_excel(self.InputFile, sheet_name ='PTF_CODES')
        self.Ptfcodes =list(self.PTF_CODES.loc[self.PTF_CODES['Selezione']==1]['PTF_CODE'] + " Client")


        self.BLP_FIELDS = pd.read_excel(self.InputFile, sheet_name ='FIELDS')
        self.Fields=list(self.BLP_FIELDS['Fields'])


    def Get_BLP_Data(self):

        DF = blp.bds(self.Ptfcodes , "PORTFOLIO_DATA", True)
        DF['PTF_CODE'] = DF.index
        DF['key']=DF['security'].map(self.remove)
        self.DF_PORTFOLIOS= DF

        self.Securities = list(DF.security.unique())
        DF_SECURITIES = blp.bdp(self.Securities, self.Fields)
        DF_SECURITIES['key']= DF_SECURITIES.index
        DF_SECURITIES['key']= DF_SECURITIES['key'].map(self.remove)
        self.DF_SECURITIES = DF_SECURITIES

        self.DF_PTF_ALL =DF_SECURITIES.merge(DF, left_on='key', right_on='key')

    def Export(this, ExportDir = None):
          if (ExportDir == None) :
                ExportDir = this.wdir

          this.DF_PTF_ALL.to_excel(ExportDir +r'\DF_PORTFOLIOS.xlsx')


In [ ]:
# We set the input value for the string variables containing
# the path to the input files.
WDIR = r'C:\Dati\BLP_API_DATA'
EXPORT_DIR = r'C:\Dati\BLP_API_DATA'
STR_INPUT_FILE = r'\BLP_API_INPUT.xlsx'

In [ ]:
OBJ = GM_BLP_PTF(WDIR)
OBJ.Load_Input(STR_INPUT_FILE)
OBJ.Get_BLP_Data()
OBJ.Export(EXPORT_DIR)
